In [ ]:
#--- GRAPH LEARNING WITH BASELINE GCN MODEL ---# 

In [1]:
# Baseline GCN (transductive)
import numpy as np, pandas as pd
from pathlib import Path
from scipy.sparse import load_npz
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split

# paths
ADJ_PATH   = "A_graph_age_rbf.npz"          # unweighted age graph
NODES_PATH = "age_weight_pruned_nodes.csv"  # node table aligned to adjacency row order

# configuration 
FEATURE_COLS = ["AGEP_A_scaled"]   # leakage-free features
LABEL_COL    = "ANXEV_A"
WEIGHT_COL   = "instance_weight"

HIDDEN   = 32
DROPOUT  = 0.3
LR       = 1e-2
EPOCHS   = 150
VAL_SIZE = 0.15
TEST_SIZE= 0.15
SEED     = 42
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

#Helpers 
def build_edge_index_weight(A):
    A = A.tocsr().astype(np.float32)
    A = ((A + A.T) * 0.5).tocsr()
    A.setdiag(0); A.eliminate_zeros()
    C = A.tocoo()
    # duplicate both directions for message passing
    row = np.concatenate([C.row, C.col])
    col = np.concatenate([C.col, C.row])
    data = np.concatenate([C.data, C.data]).astype(np.float32)
    edge_index = torch.tensor(np.vstack([row, col]), dtype=torch.long)
    edge_weight = torch.tensor(data, dtype=torch.float32)
    return edge_index, edge_weight, A.shape[0]

def normalize_adj_torch(edge_index, edge_weight, num_nodes):
    # \hat{A} = D^{-1/2}(A+I)D^{-1/2}
    device = edge_index.device
    loop = torch.arange(num_nodes, device=device)
    ei = torch.cat([edge_index, torch.stack([loop, loop])], dim=1)
    ew = torch.cat([edge_weight, torch.ones(num_nodes, device=device)], dim=0)
    deg = torch.zeros(num_nodes, device=device).scatter_add_(0, ei[0], ew)
    d_is = torch.pow(deg + 1e-12, -0.5)
    nw = d_is[ei[0]] * ew * d_is[ei[1]]
    return torch.sparse_coo_tensor(ei, nw, (num_nodes, num_nodes)).coalesce()

class GCN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim=1, dropout=0.3):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden, bias=False)
        self.lin2 = nn.Linear(hidden, out_dim, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x, A_hat):
        x = torch.sparse.mm(A_hat, x)
        x = torch.relu(self.lin1(x))
        x = self.drop(x)
        x = torch.sparse.mm(A_hat, x)
        x = self.lin2(x)
        return x.squeeze(-1)  # logits

def bce_weighted(logits, y_true, sample_weight=None, pos_weight=None):
    loss = F.binary_cross_entropy_with_logits(
        logits, y_true.float(), reduction='none', pos_weight=pos_weight
    )
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

# load data 
A = load_npz(ADJ_PATH)
edge_index, edge_weight, n = build_edge_index_weight(A)
idx_all = np.arange(n, dtype=int)

nodes = pd.read_csv(NODES_PATH)
if "node_id" in nodes.columns:
    nodes = nodes.sort_values("node_id").reset_index(drop=True)
assert len(nodes) == n, f"Adj has {n} nodes; nodes CSV has {len(nodes)} rows."

nodes[LABEL_COL]  = pd.to_numeric(nodes[LABEL_COL], errors="coerce").fillna(0).astype(int)
nodes[WEIGHT_COL] = pd.to_numeric(nodes[WEIGHT_COL], errors="coerce").fillna(0.0).astype(float)

X = nodes[FEATURE_COLS].astype(np.float32).to_numpy()
y = nodes[LABEL_COL].to_numpy(np.int64)
w = nodes[WEIGHT_COL].to_numpy(np.float32)

# transductive split (labels used only for stratification)

idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(
    idx_all, y, test_size=VAL_SIZE+TEST_SIZE, stratify=y, random_state=SEED)
rel_test = TEST_SIZE/(VAL_SIZE+TEST_SIZE)
idx_va, idx_te, y_va, y_te = train_test_split(
    idx_tmp, y_tmp, test_size=rel_test, stratify=y_tmp, random_state=SEED)

train_mask = torch.zeros(n, dtype=torch.bool); train_mask[idx_tr] = True
val_mask   = torch.zeros(n, dtype=torch.bool); val_mask[idx_va] = True
test_mask  = torch.zeros(n, dtype=torch.bool); test_mask[idx_te] = True

#Tensors

xT  = torch.from_numpy(X).to(DEVICE)
yT  = torch.from_numpy(y).to(DEVICE)
swT = torch.from_numpy(w).to(DEVICE)
eiT = edge_index.to(DEVICE)
ewT = edge_weight.to(DEVICE)

# normalized adjacency on the FULL graph (transductive)
A_hat = normalize_adj_torch(eiT, ewT, n)

#  model 
model = GCN(xT.size(1), HIDDEN, out_dim=1, dropout=DROPOUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

# class imbalance weight for BCE
pos_w = torch.tensor([(y==0).sum()/max(1,(y==1).sum())], dtype=torch.float32, device=DEVICE)

# train (pick best by validation loss; no metrics) 
best_val = np.inf
best_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    logits = model(xT, A_hat)
    loss_tr = bce_weighted(logits[train_mask.to(DEVICE)], yT[train_mask.to(DEVICE)],
                           swT[train_mask.to(DEVICE)], pos_w)

    opt.zero_grad()
    loss_tr.backward()
    opt.step()

    # validation loss 
    model.eval()
    with torch.no_grad():
        logits_val = model(xT, A_hat)
        loss_val = bce_weighted(logits_val[val_mask.to(DEVICE)], yT[val_mask.to(DEVICE)],
                                swT[val_mask.to(DEVICE)], pos_w)

    if loss_val.item() < best_val:
        best_val = loss_val.item()
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f"Ep {epoch:03d} | train_loss={loss_tr.item():.4f} | val_loss={loss_val.item():.4f}")

# load best-by-val-loss weights
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

#  artifacts to reuse for evaluation  
# model, A_hat, xT, yT, swT, train_mask, val_mask, test_mask, nodes
print("Done. Model trained (transductive). Ready for separate evaluation.")


Ep 001 | train_loss=1.1351 | val_loss=1.1220
Ep 010 | train_loss=1.0874 | val_loss=1.0761
Ep 020 | train_loss=1.0869 | val_loss=1.0740
Ep 030 | train_loss=1.0849 | val_loss=1.0730
Ep 040 | train_loss=1.0848 | val_loss=1.0742
Ep 050 | train_loss=1.0848 | val_loss=1.0738
Ep 060 | train_loss=1.0844 | val_loss=1.0733
Ep 070 | train_loss=1.0848 | val_loss=1.0733
Ep 080 | train_loss=1.0847 | val_loss=1.0735
Ep 090 | train_loss=1.0847 | val_loss=1.0734
Ep 100 | train_loss=1.0844 | val_loss=1.0734
Ep 110 | train_loss=1.0847 | val_loss=1.0734
Ep 120 | train_loss=1.0845 | val_loss=1.0734
Ep 130 | train_loss=1.0846 | val_loss=1.0734
Ep 140 | train_loss=1.0845 | val_loss=1.0734
Ep 150 | train_loss=1.0844 | val_loss=1.0734
Done. Model trained (transductive). Ready for separate evaluation.


In [13]:
#  Efficiency Evaluation: Accuracy, PR-AUC (AP), F1 
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score, average_precision_score

# test predictions
model.eval()
with torch.no_grad():
    logits = model(xT, A_hat)
probs_all = torch.sigmoid(logits).cpu().numpy()
preds_all = (probs_all >= 0.5).astype(int)

y_all     = yT.cpu().numpy()
test_idx  = np.where(test_mask.cpu().numpy())[0]

# Efficiency (test)
acc = accuracy_score(y_all[test_idx], preds_all[test_idx])
ap  = average_precision_score(y_all[test_idx], probs_all[test_idx])  # PR-AUC (AP)
f1p = f1_score(y_all[test_idx], preds_all[test_idx], pos_label=1)

print("=== Efficiency (Test) ===")
print(f"Accuracy : {acc:.3f}")
print(f"PR-AUC(AP): {ap:.3f}")
print(f"F1(+1)   : {f1p:.3f}")

# (optional) show prevalence baseline for context
print(f"(Positive prevalence in TEST: {y_all[test_idx].mean():.3f})")




=== Efficiency (Test) ===
Accuracy : 0.810
PR-AUC(AP): 0.222
F1(+1)   : 0.007
(Positive prevalence in TEST: 0.184)


In [3]:
#Fairness Evaluation: DI and SPD

import numpy as np, pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

test_idx = np.where(test_mask.cpu().numpy())[0]
y_true   = yT.cpu().numpy()[test_idx].astype(int)
y_pred   = ((probs_all >= 0.5).astype(int))[test_idx]

# survey weights for weighted SPD/DI
w_test = pd.to_numeric(nodes.loc[test_idx, "instance_weight"], errors="coerce").fillna(0.0).to_numpy(float)

def aif_spd_di(attr_col, privileged_value=1, favorable_label=1, unfavorable_label=0):
    """
    Computes SPD & DI using AIF360 for a binary protected attribute.
    Map: 0 = unprivileged, 1 = privileged  -> privileged_value=1
    """
    g = pd.to_numeric(nodes.loc[test_idx, attr_col], errors="coerce").fillna(0).astype(int).to_numpy()

    # Build AIF360 datasets (unweighted)
    df_true = pd.DataFrame({attr_col: g, "label": y_true})
    df_pred = pd.DataFrame({attr_col: g, "label": y_pred})
    ds_true_unw = BinaryLabelDataset(
        favorable_label=favorable_label, unfavorable_label=unfavorable_label,
        df=df_true, label_names=["label"], protected_attribute_names=[attr_col]
    )
    ds_pred_unw = BinaryLabelDataset(
        favorable_label=favorable_label, unfavorable_label=unfavorable_label,
        df=df_pred, label_names=["label"], protected_attribute_names=[attr_col]
    )

    # Weighted copies
    ds_true_w = ds_true_unw.copy(); ds_true_w.instance_weights = w_test.copy()
    ds_pred_w = ds_pred_unw.copy(); ds_pred_w.instance_weights = w_test.copy()

    # Group definitions: unprivileged = 0, privileged = 1
    unprivileged_groups = [{attr_col: 0}]  #SEX_A >> male and HISPALLP_BIN >> Hispanic + other minorities
    privileged_groups   = [{attr_col: privileged_value}]  # (=1) SEX_A >> female and HISPALLP_BIN >> Non-Hispanic White-only

    cm_unw = ClassificationMetric(ds_true_unw, ds_pred_unw,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)
    cm_w   = ClassificationMetric(ds_true_w,  ds_pred_w,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)

    spd_unw = cm_unw.statistical_parity_difference()   # unpriv - priv
    di_unw  = cm_unw.disparate_impact()               # unpriv / priv
    spd_w   = cm_w.statistical_parity_difference()
    di_w    = cm_w.disparate_impact()

    return spd_unw, di_unw, spd_w, di_w

rows = []

for attr, label in [
    ("SEX_A", "SEX_A (0=Male unpriv, 1=Female priv)"),
    ("HISPALLP_BIN", "HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)")
]:
    if attr not in nodes.columns:
        continue
    spd_unw, di_unw, spd_w, di_w = aif_spd_di(attr, privileged_value=1)
    rows.append({
        "attr": label,
        "SPD_unw": round(spd_unw, 3),
        "DI_unw":  round(di_unw, 3) if np.isfinite(di_unw) else np.nan,
        "SPD_w":   round(spd_w,   3),
        "DI_w":    round(di_w,   3) if np.isfinite(di_w) else np.nan,
        "4/5ths_ok_unw": (0.8 <= di_unw <= 1.25) if np.isfinite(di_unw) else np.nan,
        "4/5ths_ok_w":   (0.8 <= di_w   <= 1.25) if np.isfinite(di_w)   else np.nan,
    })

fair_aif = pd.DataFrame(rows)
print("=== AIF360 SPD / DI (Test) with the privilege mapping ===")
print(fair_aif.to_string(index=False))




2025-09-01 01:14:07.487884: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756685647.605299   58378 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756685647.639261   58378 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756685647.876516   58378 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756685647.876598   58378 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756685647.876601   58378 computation_placer.cc:177] computation placer alr

=== AIF360 SPD / DI (Test) with the privilege mapping ===
                                                attr  SPD_unw  DI_unw  SPD_w  DI_w  4/5ths_ok_unw  4/5ths_ok_w
                SEX_A (0=Male unpriv, 1=Female priv)    0.001   1.197  0.001 1.120           True         True
HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)    0.004   1.692  0.003 1.577          False        False


In [5]:
#  Homophily summary (SEX_A, HISPALLP_BIN, ANXEV_A) — Global and Test subgraph 

import numpy as np, pandas as pd
from pathlib import Path
from scipy.sparse import load_npz, csr_matrix, triu

#  load adjacency in SciPy 
def load_adj_scipy():
    
    try:
        return A.copy().tocsr()
    except NameError:
        pass
    
    adj_path = "A_graph_age_rbf.npz" if Path("A_graph_age_rbf.npz").exists() else "age_weight_pruned_adjacency.npz"
    return load_npz(adj_path).tocsr()

A_s = load_adj_scipy().astype(np.float64).tocsr()
A_s.setdiag(0.0); A_s.eliminate_zeros()
A_s = ((A_s + A_s.T) * 0.5).tocsr()  # ensure symmetry

#load node attributes and align to row order 
nodes = pd.read_csv("age_weight_pruned_nodes.csv")
if "node_id" in nodes.columns:
    nodes = nodes.sort_values("node_id").reset_index(drop=True)

# binaries -> {0,1}
for c in ["SEX_A","HISPALLP_BIN","ANXEV_A"]:
    if c in nodes.columns:
        nodes[c] = pd.to_numeric(nodes[c], errors="coerce").fillna(0).astype(int)

def homophily_from_A(A_csr: csr_matrix, group01: np.ndarray):
    U = triu(A_csr, k=1).tocoo()
    u, v, w = U.row, U.col, U.data
    if w.size == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    # observed edge-weighted (weighted by edge weight w)
    same = (group01[u] == group01[v])
    obs = float(w[same].sum() / (w.sum() + 1e-12))
    # expected baseline (endpoint shares; unweighted)
    endpoints = np.concatenate([group01[u], group01[v]])
    counts = np.bincount(endpoints, minlength=2).astype(float)
    p = counts / counts.sum()
    exp = float((p**2).sum())
    delta = obs - exp
    # degree & exposure
    deg = np.asarray(A_csr.sum(1)).ravel()
    d0 = float(deg[group01==0].mean()) if (group01==0).any() else np.nan
    d1 = float(deg[group01==1].mean()) if (group01==1).any() else np.nan
    deg_eps = deg + 1e-12
    frac1 = np.asarray(A_csr.dot(group01)) / deg_eps   # share of incident weight to group==1
    same_exp = np.where(group01==1, frac1, 1.0 - frac1)
    e0 = float(same_exp[group01==0].mean()) if (group01==0).any() else np.nan
    e1 = float(same_exp[group01==1].mean()) if (group01==1).any() else np.nan
    return obs, exp, delta, d0, d1, e0, e1

def print_block(title, A_csr, nodes_df, col, labels):
    g = nodes_df[col].to_numpy(int)
    obs, exp, delta, d0, d1, e0, e1 = homophily_from_A(A_csr, g)
    print(f"{title}: homophily obs={obs:.3f} exp={exp:.3f} Δ={delta:+.3f}")
    print(f"  mean degree: {labels[0]}={d0:.2f}, {labels[1]}={d1:.2f}")
    print(f"  same-group exposure: {labels[0]}={e0:.3f}, {labels[1]}={e1:.3f}")

#  Global graph 
print("=== Homophily — Global graph ===")
if "SEX_A" in nodes.columns:
    print_block("SEX",  A_s, nodes, "SEX_A", ("Male","Female"))
if "HISPALLP_BIN" in nodes.columns:
    print_block("HISP", A_s, nodes, "HISPALLP_BIN", ("Hisp+Others","Non-Hisp White"))
if "ANXEV_A" in nodes.columns:
    print_block("ANXEV_A", A_s, nodes, "ANXEV_A", ("No","Yes"))

# Test subgraph (uses transductive split) 
try:
    test_idx = np.where(test_mask.cpu().numpy())[0]
    A_test = A_s[test_idx][:, test_idx].tocsr()
    nodes_test = nodes.iloc[test_idx].reset_index(drop=True)
    print("\n=== Homophily — Test subgraph only ===")
    if "SEX_A" in nodes_test.columns:
        print_block("SEX (test)",  A_test, nodes_test, "SEX_A", ("Male","Female"))
    if "HISPALLP_BIN" in nodes_test.columns:
        print_block("HISP (test)", A_test, nodes_test, "HISPALLP_BIN", ("Hisp+Others","Non-Hisp White"))
    if "ANXEV_A" in nodes_test.columns:
        print_block("ANXEV_A (test)", A_test, nodes_test, "ANXEV_A", ("No","Yes"))
except NameError:
    print("\n[Note] test_mask not found — printed global homophily only. Run after your GCN split to get test-only.")


=== Homophily — Global graph ===
SEX: homophily obs=0.504 exp=0.503 Δ=+0.001
  mean degree: Male=30.27, Female=29.77
  same-group exposure: Male=0.464, Female=0.537
HISP: homophily obs=0.567 exp=0.550 Δ=+0.018
  mean degree: Hisp+Others=30.29, Non-Hisp White=29.85
  same-group exposure: Hisp+Others=0.373, Non-Hisp White=0.667
ANXEV_A: homophily obs=0.706 exp=0.703 Δ=+0.003
  mean degree: No=30.11, Yes=29.50
  same-group exposure: No=0.823, Yes=0.187

=== Homophily — Test subgraph only ===
SEX (test): homophily obs=0.494 exp=0.501 Δ=-0.007
  mean degree: Male=4.60, Female=4.15
  same-group exposure: Male=0.519, Female=0.465
HISP (test): homophily obs=0.572 exp=0.553 Δ=+0.019
  mean degree: Hisp+Others=4.32, Non-Hisp White=4.38
  same-group exposure: Hisp+Others=0.368, Non-Hisp White=0.647
ANXEV_A (test): homophily obs=0.713 exp=0.706 Δ=+0.006
  mean degree: No=4.39, Yes=4.23
  same-group exposure: No=0.844, Yes=0.171


In [6]:
#  Imbalance-aware GCN training (survey-weighted) 

import numpy as np, torch, torch.nn.functional as F
from sklearn.metrics import precision_recall_curve, average_precision_score

# config toggles 
USE_FOCAL     = True      # set False to use weighted BCE only
FOCAL_GAMMA   = 2.0       # typical: 1.5–2.0
FOCAL_ALPHA   = None      # None -> auto from train prevalence; else float in [0,1]
EPOCHS        = 200
LR            = 1e-2
SEED          = 42
torch.manual_seed(SEED)

# helpers 
def train_prevalence(y, w, mask):
    idx = mask.cpu().numpy().astype(bool)
    yw  = (y[idx].cpu().numpy() * w[idx].cpu().numpy()).sum()
    ww  = w[idx].cpu().numpy().sum()
    piw = float(yw / max(ww, 1e-12))   # weighted prevalence in train
    return piw

def bce_weighted_loss(logits, y_true, sample_weight, pos_weight_scalar):
    loss = F.binary_cross_entropy_with_logits(
        logits, y_true.float(), reduction='none',
        pos_weight=torch.tensor([pos_weight_scalar], device=logits.device)
    )
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def focal_binary_loss(logits, y_true, sample_weight, gamma=2.0, alpha=None):

    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    y = y_true.float()
    # per-sample BCE
    bce = F.binary_cross_entropy(p, y, reduction="none")
    # focal modulating factor
    pt = torch.where(y > 0.5, p, 1 - p)
    focal = (1 - pt) ** gamma
    # alpha balance
    if alpha is None:
        # set alpha ≈ proportion of negative class (common in focal loss) to increase weight on positives when positives are rare
        pos_frac = y.mean().item() if y.numel() > 0 else 0.5
        alpha = 1.0 - pos_frac
    alpha_t = torch.where(y > 0.5, torch.tensor(alpha, device=logits.device),
                          torch.tensor(1.0 - alpha, device=logits.device))
    loss = alpha_t * focal * bce
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def pick_tau_on_val(y_val, p_val):
    # Choose τ* maximizing F1 on validation
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2*prec*rec / (prec+rec+1e-12)
    best = int(np.nanargmax(f1))
    tau = thr[max(0, min(best, len(thr)-1))]
    return float(tau), float(f1[best]), float(average_precision_score(y_val, p_val))

#  compute survey-weighted pos_weight on TRAIN 
pi_w = train_prevalence(yT, swT, train_mask)     # e.g., 0.184 (weighted)
pos_w = (1.0 - pi_w) / max(pi_w, 1e-12)          # class weight for positive class
print(f"Train weighted prevalence π_w={pi_w:.3f} -> pos_weight={pos_w:.3f}")

# (re)initialize model & optimizer 
torch.manual_seed(SEED)
model = GCN(xT.size(1), HIDDEN, out_dim=1, dropout=DROPOUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

#  training loop (select best by validation AUPR) 
best_aupr, best_state = -1.0, None

for epoch in range(1, EPOCHS+1):
    model.train()
    logits = model(xT, A_hat)

    if USE_FOCAL:
        loss_tr = focal_binary_loss(
            logits[train_mask], yT[train_mask], swT[train_mask],
            gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA
        )
    else:
        loss_tr = bce_weighted_loss(
            logits[train_mask], yT[train_mask], swT[train_mask], pos_w
        )

    opt.zero_grad()
    loss_tr.backward()
    opt.step()

    # validation PR metrics for selection
    model.eval()
    with torch.no_grad():
        logits_val = model(xT, A_hat)[val_mask]
        probs_val  = torch.sigmoid(logits_val).cpu().numpy()
        y_val      = yT[val_mask].cpu().numpy()
        tau_val, f1_val, aupr_val = pick_tau_on_val(y_val, probs_val)

    if aupr_val > best_aupr:
        best_aupr = aupr_val
        best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
        best_info = dict(epoch=epoch, tau=tau_val, f1=f1_val, aupr=aupr_val)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Ep {epoch:03d} | train_loss={loss_tr.item():.4f} | val_AUPR={aupr_val:.3f} | τ*={tau_val:.3f} | F1_val={f1_val:.3f}")

# restore best-by-AUPR weights
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k,v in best_state.items()})
    print(f"Selected epoch {best_info['epoch']} (val AUPR={best_info['aupr']:.3f}, τ*={best_info['tau']:.3f}, F1_val={best_info['f1']:.3f})")
else:
    print("Warning: no best_state captured; keeping last epoch weights.")

# save the chosen threshold for downstream evaluation 
TAU_STAR = best_info['tau'] if 'best_info' in locals() else 0.5
print(f"TAU_STAR (use on TEST): {TAU_STAR:.3f}")




Train weighted prevalence π_w=0.169 -> pos_weight=4.909
Ep 001 | train_loss=0.0575 | val_AUPR=0.150 | τ*=0.502 | F1_val=0.311
Ep 010 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.465 | F1_val=0.336
Ep 020 | train_loss=0.0505 | val_AUPR=0.226 | τ*=0.453 | F1_val=0.336
Ep 030 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.482 | F1_val=0.336
Ep 040 | train_loss=0.0501 | val_AUPR=0.226 | τ*=0.483 | F1_val=0.336
Ep 050 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.472 | F1_val=0.336
Ep 060 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.474 | F1_val=0.336
Ep 070 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.478 | F1_val=0.336
Ep 080 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.475 | F1_val=0.336
Ep 090 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.475 | F1_val=0.336
Ep 100 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.476 | F1_val=0.336
Ep 110 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.476 | F1_val=0.336
Ep 120 | train_loss=0.0500 | val_AUPR=0.226 | τ*=0.476 | F1_val=0.336
Ep 130 | train_loss=0.0500 | val_A

In [14]:
# Efficiency Evaluation: Accuracy, PR-AUC (AP), F1 (after Imbalance-aware GCN) 
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score, average_precision_score

# test predictions 
model.eval()
with torch.no_grad():
    logits = model(xT, A_hat)
probs_all = torch.sigmoid(logits).cpu().numpy()

# use tuned threshold from validation
tau = globals().get("TAU_STAR", 0.5)
preds_all = (probs_all >= tau).astype(int)

y_all     = yT.cpu().numpy()
test_idx  = np.where(test_mask.cpu().numpy())[0]

# Efficiency (test)
acc = accuracy_score(y_all[test_idx], preds_all[test_idx])
ap  = average_precision_score(y_all[test_idx], probs_all[test_idx])  # PR-AUC (AP)
f1p = f1_score(y_all[test_idx], preds_all[test_idx], pos_label=1)

print("=== Efficiency (Test) ===")
print(f"Accuracy : {acc:.3f}")
print(f"PR-AUC(AP): {ap:.3f}")
print(f"F1(+1)   : {f1p:.3f}")
print(f"(Test prevalence: {y_all[test_idx].mean():.3f})")


=== Efficiency (Test) ===
Accuracy : 0.467
PR-AUC(AP): 0.222
F1(+1)   : 0.329
(Test prevalence: 0.184)


In [8]:
#Fairness Evaluation: DI and SPD (after Imbalance-aware GCN)

import numpy as np, pandas as pd
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

test_idx = np.where(test_mask.cpu().numpy())[0]
y_true   = yT.cpu().numpy()[test_idx].astype(int)
y_pred   = ((probs_all >= TAU_STAR).astype(int))[test_idx]

# survey weights for weighted SPD/DI
w_test = pd.to_numeric(nodes.loc[test_idx, "instance_weight"], errors="coerce").fillna(0.0).to_numpy(float)

def aif_spd_di(attr_col, privileged_value=1, favorable_label=1, unfavorable_label=0):
    """
    Computes SPD & DI using AIF360 for a binary protected attribute.
    Map: 0 = unprivileged, 1 = privileged  -> privileged_value=1
    """
    g = pd.to_numeric(nodes.loc[test_idx, attr_col], errors="coerce").fillna(0).astype(int).to_numpy()

    # Build AIF360 datasets (unweighted)
    df_true = pd.DataFrame({attr_col: g, "label": y_true})
    df_pred = pd.DataFrame({attr_col: g, "label": y_pred})
    ds_true_unw = BinaryLabelDataset(
        favorable_label=favorable_label, unfavorable_label=unfavorable_label,
        df=df_true, label_names=["label"], protected_attribute_names=[attr_col]
    )
    ds_pred_unw = BinaryLabelDataset(
        favorable_label=favorable_label, unfavorable_label=unfavorable_label,
        df=df_pred, label_names=["label"], protected_attribute_names=[attr_col]
    )

    # Weighted copies
    ds_true_w = ds_true_unw.copy(); ds_true_w.instance_weights = w_test.copy()
    ds_pred_w = ds_pred_unw.copy(); ds_pred_w.instance_weights = w_test.copy()

    # Group definitions: unprivileged = 0, privileged = 1
    unprivileged_groups = [{attr_col: 0}]  #SEX_A >> male and HISPALLP_BIN >> Hispanic + other minorities
    privileged_groups   = [{attr_col: privileged_value}]  # (=1) SEX_A >> female and HISPALLP_BIN >> Non-Hispanic White-only

    cm_unw = ClassificationMetric(ds_true_unw, ds_pred_unw,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)
    cm_w   = ClassificationMetric(ds_true_w,  ds_pred_w,
                                  unprivileged_groups=unprivileged_groups,
                                  privileged_groups=privileged_groups)

    spd_unw = cm_unw.statistical_parity_difference()   # unpriv - priv
    di_unw  = cm_unw.disparate_impact()               # unpriv / priv
    spd_w   = cm_w.statistical_parity_difference()
    di_w    = cm_w.disparate_impact()

    return spd_unw, di_unw, spd_w, di_w

rows = []

for attr, label in [
    ("SEX_A", "SEX_A (0=Male unpriv, 1=Female priv)"),
    ("HISPALLP_BIN", "HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)")
]:
    if attr not in nodes.columns:
        continue
    spd_unw, di_unw, spd_w, di_w = aif_spd_di(attr, privileged_value=1)
    rows.append({
        "attr": label,
        "SPD_unw": round(spd_unw, 3),
        "DI_unw":  round(di_unw, 3) if np.isfinite(di_unw) else np.nan,
        "SPD_w":   round(spd_w,   3),
        "DI_w":    round(di_w,   3) if np.isfinite(di_w) else np.nan,
        "4/5ths_ok_unw": (0.8 <= di_unw <= 1.25) if np.isfinite(di_unw) else np.nan,
        "4/5ths_ok_w":   (0.8 <= di_w   <= 1.25) if np.isfinite(di_w)   else np.nan,
    })

fair_aif = pd.DataFrame(rows)
print("=== AIF360 SPD / DI (Test) with the privilege mapping ===")
print(fair_aif.to_string(index=False))


=== AIF360 SPD / DI (Test) with the privilege mapping ===
                                                attr  SPD_unw  DI_unw  SPD_w  DI_w  4/5ths_ok_unw  4/5ths_ok_w
                SEX_A (0=Male unpriv, 1=Female priv)    0.026   1.043  0.039 1.066           True         True
HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)    0.188   1.345  0.200 1.373          False        False


In [15]:
# === FairDrop sweep (HISP-only) — using PR-AUC (AP) ===
import numpy as np, pandas as pd, torch
from sklearn.metrics import precision_recall_curve, average_precision_score, accuracy_score, f1_score

DEVICE = next(model.parameters()).device  # reuse device
SEED = 42
USE_FOCAL   = True
FOCAL_GAMMA = 2.0
EPOCHS      = 200
LR          = 1e-2

# helper losses
def bce_weighted_loss(logits, y_true, sample_weight, pos_weight_scalar):
    import torch.nn.functional as F
    loss = F.binary_cross_entropy_with_logits(
        logits, y_true.float(), reduction='none',
        pos_weight=torch.tensor([pos_weight_scalar], device=logits.device)
    )
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def focal_binary_loss(logits, y_true, sample_weight, gamma=2.0, alpha=None):
    import torch.nn.functional as F
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    y = y_true.float()
    bce = F.binary_cross_entropy(p, y, reduction="none")
    pt = torch.where(y > 0.5, p, 1 - p)
    focal = (1 - pt) ** gamma
    if alpha is None:
        pos_frac = y.mean().item() if y.numel() > 0 else 0.5
        alpha = 1.0 - pos_frac
    alpha_t = torch.where(y > 0.5, torch.tensor(alpha, device=logits.device),
                          torch.tensor(1.0 - alpha, device=logits.device))
    loss = alpha_t * focal * bce
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def pick_tau_on_val(y_val, p_val):
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2*prec*rec / (prec+rec+1e-12)
    bi = int(np.nanargmax(f1))
    tau = thr[max(0, min(bi, len(thr)-1))]
    return float(tau), float(f1[bi]), float(average_precision_score(y_val, p_val))

def build_fairdrop_train_adj_HISP(p_same=0.1):
    """Drop only edges where HISPALLP_BIN is the same at both endpoints; keep eval adjacency unchanged."""
    g = torch.from_numpy(pd.to_numeric(nodes["HISPALLP_BIN"], errors="coerce").fillna(0).astype(int).to_numpy()).to(DEVICE)
    src, dst = eiT[0], eiT[1]
    same = (g[src] == g[dst])
    keep_prob = torch.where(same, torch.tensor(1.0 - p_same, device=DEVICE), torch.tensor(1.0, device=DEVICE))
    keep_mask = torch.rand_like(ewT) < keep_prob
    ei_train = eiT[:, keep_mask]
    ew_train = ewT[keep_mask]
    return normalize_adj_torch(ei_train, ew_train, n)  # training adjacency
    # evaluation uses the full A_hat (no drops)

def train_val_select(A_hat_train, A_hat_eval, pos_weight_scalar):
    torch.manual_seed(SEED)
    m = GCN(xT.size(1), HIDDEN, out_dim=1, dropout=DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    best_aupr, best_state, best_info = -1.0, None, None
    for ep in range(1, EPOCHS+1):
        m.train()
        logits = m(xT, A_hat_train)
        if USE_FOCAL:
            loss = focal_binary_loss(logits[train_mask], yT[train_mask], swT[train_mask], gamma=FOCAL_GAMMA)
        else:
            loss = bce_weighted_loss(logits[train_mask], yT[train_mask], swT[train_mask], pos_weight_scalar)
        opt.zero_grad(); loss.backward(); opt.step()
        # val AUPR selection
        m.eval()
        with torch.no_grad():
            pv = torch.sigmoid(m(xT, A_hat_eval)[val_mask]).cpu().numpy()
            yv = yT[val_mask].cpu().numpy()
            tau, f1v, aupr = pick_tau_on_val(yv, pv)
        if aupr > best_aupr:
            best_aupr, best_state = aupr, {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
            best_info = dict(epoch=ep, tau=tau, f1=f1v, aupr=aupr)
    if best_state is not None:
        m.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})
    return m, best_info

def eval_test(m, A_hat_eval, tau):
    with torch.no_grad():
        p = torch.sigmoid(m(xT, A_hat_eval)).cpu().numpy()
    pr = (p >= tau).astype(int)
    y  = yT.cpu().numpy()
    te = np.where(test_mask.cpu().numpy())[0]
    acc = accuracy_score(y[te], pr[te])
    ap  = average_precision_score(y[te], p[te])    # <-- PR-AUC (AP)
    f1p = f1_score(y[te], pr[te], pos_label=1)
    return acc, ap, f1p, pr, p

def fairness_DI_SPD(preds):
    rows = []
    test_idx = np.where(test_mask.cpu().numpy())[0]
    sw = pd.to_numeric(nodes.loc[test_idx, "instance_weight"], errors="coerce").fillna(0.0).to_numpy(float)
    for col, pretty in [("SEX_A","SEX_A"),("HISPALLP_BIN","HISPALLP_BIN")]:
        if col not in nodes.columns: continue
        g = pd.to_numeric(nodes.loc[test_idx, col], errors="coerce").fillna(0).astype(int).to_numpy()
        g0 = (g==0); g1 = (g==1)
        p0  = preds[test_idx][g0].mean() if g0.any() else np.nan
        p1  = preds[test_idx][g1].mean() if g1.any() else np.nan
        p0w = np.average(preds[test_idx][g0], weights=sw[g0]) if g0.any() and sw[g0].sum()>0 else np.nan
        p1w = np.average(preds[test_idx][g1], weights=sw[g1]) if g1.any() and sw[g1].sum()>0 else np.nan
        spd   = p0 - p1
        di    = (p0 / p1) if (p1 not in [0, np.nan]) else np.nan
        spd_w = p0w - p1w
        di_w  = (p0w / p1w) if (p1w not in [0, np.nan]) else np.nan
        rows.append({"attr":pretty, "SPD":spd, "DI":di, "SPD_w":spd_w, "DI_w":di_w})
    return pd.DataFrame(rows)

# compute train-weighted pos_weight once 
pi_w = float((yT[train_mask] * swT[train_mask]).sum().cpu().item() / max(swT[train_mask].sum().cpu().item(), 1e-12))
pos_w = (1.0 - pi_w) / max(pi_w, 1e-12)

# Baseline row (PR-AUC) 
with torch.no_grad():
    pv = torch.sigmoid(model(xT, A_hat)[val_mask]).cpu().numpy()
    yv = yT[val_mask].cpu().numpy()
TAU_STAR, F1v, AUPRv = pick_tau_on_val(yv, pv)

acc0, ap0, f10, pr0, p0 = eval_test(model, A_hat, TAU_STAR)
base_fair = fairness_DI_SPD(pr0)
print("\nBaseline (no FairDrop): "
      f"Acc={acc0:.3f} PR-AUC={ap0:.3f} F1={f10:.3f} | "
      f"HISP DI_w={base_fair.loc[base_fair.attr=='HISPALLP_BIN','DI_w'].values[0]:.3f}, "
      f"SPD_w={base_fair.loc[base_fair.attr=='HISPALLP_BIN','SPD_w'].values[0]:+.3f}")

#  FairDrop sweep over p_same (HISP-only) 
rows = []
for p_same in [0.05, 0.10, 0.20]:
    A_hat_train = build_fairdrop_train_adj_HISP(p_same=p_same)  # train on dropped adj
    A_hat_eval  = A_hat                                        # evaluate on full adj
    m, info = train_val_select(A_hat_train, A_hat_eval, pos_w)
    acc, ap, f1p, pr, p = eval_test(m, A_hat_eval, info['tau'])
    fair = fairness_DI_SPD(pr)
    rows.append({
        "method": f"FairDrop(HISP, p={p_same})",
        "val_AUPR": info['aupr'], "tau*": info['tau'],
        "Acc": acc, "PR-AUC": ap, "F1+": f1p,
        "DI_w(HISP)": float(fair.loc[fair.attr=="HISPALLP_BIN","DI_w"].values[0]),
        "SPD_w(HISP)": float(fair.loc[fair.attr=="HISPALLP_BIN","SPD_w"].values[0]),
    })

comp = pd.DataFrame(rows).round(3)
print("\n=== FairDrop(HISP-only) comparison (PR-AUC) ===")
print(comp.to_string(index=False))





Baseline (no FairDrop): Acc=0.467 PR-AUC=0.222 F1=0.329 | HISP DI_w=1.373, SPD_w=+0.200

=== FairDrop(HISP-only) comparison (PR-AUC) ===
                method  val_AUPR  tau*   Acc  PR-AUC   F1+  DI_w(HISP)  SPD_w(HISP)
FairDrop(HISP, p=0.05)     0.226 0.476 0.467   0.222 0.329       1.373          0.2
 FairDrop(HISP, p=0.1)     0.226 0.476 0.467   0.222 0.329       1.373          0.2
 FairDrop(HISP, p=0.2)     0.226 0.476 0.467   0.222 0.329       1.373          0.2


In [17]:
# Inspect how many edges are HISP-same and how many were dropped
import torch, numpy as np, pandas as pd

g_hisp = torch.from_numpy(pd.to_numeric(nodes["HISPALLP_BIN"], errors="coerce").fillna(0).astype(int).to_numpy()).to(DEVICE)
src, dst = eiT[0], eiT[1]
is_same = (g_hisp[src] == g_hisp[dst]).cpu().numpy()

def drop_stats(p_same):
    keep_prob = np.where(is_same, 1.0 - p_same, 1.0)
    rng = np.random.default_rng(0)
    keep = rng.random(keep_prob.shape) < keep_prob
    kept_same = keep & is_same
    kept_cross = keep & (~is_same)
    return dict(
        total=len(keep),
        same_edges=int(is_same.sum()),
        cross_edges=int((~is_same).sum()),
        kept_same=int(kept_same.sum()),
        kept_cross=int(kept_cross.sum()),
        drop_rate_same=1 - kept_same.sum()/max(is_same.sum(),1),
        drop_rate_cross=1 - kept_cross.sum()/max((~is_same).sum(),1),
    )

for ps in [0.05, 0.10, 0.20, 0.50]:
    print(ps, drop_stats(ps))


0.05 {'total': 3406800, 'same_edges': 1932720, 'cross_edges': 1474080, 'kept_same': 1836094, 'kept_cross': 1474080, 'drop_rate_same': np.float64(0.04999482594478244), 'drop_rate_cross': np.float64(0.0)}
0.1 {'total': 3406800, 'same_edges': 1932720, 'cross_edges': 1474080, 'kept_same': 1739509, 'kept_cross': 1474080, 'drop_rate_same': np.float64(0.09996843826317314), 'drop_rate_cross': np.float64(0.0)}
0.2 {'total': 3406800, 'same_edges': 1932720, 'cross_edges': 1474080, 'kept_same': 1546543, 'kept_cross': 1474080, 'drop_rate_same': np.float64(0.19981011217351707), 'drop_rate_cross': np.float64(0.0)}
0.5 {'total': 3406800, 'same_edges': 1932720, 'cross_edges': 1474080, 'kept_same': 965755, 'kept_cross': 1474080, 'drop_rate_same': np.float64(0.5003130303406598), 'drop_rate_cross': np.float64(0.0)}


In [16]:
# FairDrop sweep (HISP-only) with FairDrop stochastic per epoch
import numpy as np, pandas as pd, torch
from sklearn.metrics import precision_recall_curve, average_precision_score, accuracy_score, f1_score

DEVICE = next(model.parameters()).device  # reuse device
SEED = 42
USE_FOCAL   = True
FOCAL_GAMMA = 2.0
EPOCHS      = 200
LR          = 1e-2

# helper losses 
def bce_weighted_loss(logits, y_true, sample_weight, pos_weight_scalar):
    import torch.nn.functional as F
    loss = F.binary_cross_entropy_with_logits(
        logits, y_true.float(), reduction='none',
        pos_weight=torch.tensor([pos_weight_scalar], device=logits.device)
    )
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def focal_binary_loss(logits, y_true, sample_weight, gamma=2.0, alpha=None):
    import torch.nn.functional as F
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    y = y_true.float()
    bce = F.binary_cross_entropy(p, y, reduction="none")
    pt = torch.where(y > 0.5, p, 1 - p)
    focal = (1 - pt) ** gamma
    if alpha is None:
        pos_frac = y.mean().item() if y.numel() > 0 else 0.5
        alpha = 1.0 - pos_frac
    alpha_t = torch.where(y > 0.5, torch.tensor(alpha, device=logits.device),
                          torch.tensor(1.0 - alpha, device=logits.device))
    loss = alpha_t * focal * bce
    if sample_weight is not None:
        sw = sample_weight / (sample_weight.mean() + 1e-12)
        loss = loss * sw
    return loss.mean()

def pick_tau_on_val(y_val, p_val):
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2*prec*rec / (prec+rec+1e-12)
    bi = int(np.nanargmax(f1))
    tau = thr[max(0, min(bi, len(thr)-1))]
    return float(tau), float(f1[bi]), float(average_precision_score(y_val, p_val))  # AP = val_AUPR

#  FairDrop sampler (stochastic per epoch) 
def sample_train_adj_hisp(p_same):  #updating mask from previous Fairdrop
    g = torch.from_numpy(pd.to_numeric(nodes["HISPALLP_BIN"], errors="coerce").fillna(0).astype(int).to_numpy()).to(DEVICE)
    src, dst = eiT[0], eiT[1]
    same = (g[src] == g[dst])
    keep_prob = torch.where(same, torch.tensor(1.0 - p_same, device=DEVICE), torch.tensor(1.0, device=DEVICE))
    keep = torch.rand_like(ewT) < keep_prob
    ei = eiT[:, keep]; ew = ewT[keep]
    return normalize_adj_torch(ei, ew, n)

def train_with_stochastic_fairdrop(p_same, epochs=200): #updating mask from previous Fairdrop
    torch.manual_seed(SEED)
    m = GCN(xT.size(1), HIDDEN, out_dim=1, dropout=DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    best_aupr, best_state, best_info = -1.0, None, None

    for ep in range(1, epochs+1):
        m.train()
        A_hat_train = sample_train_adj_hisp(p_same)  # resample each epoch
        logits = m(xT, A_hat_train)
        if USE_FOCAL:
            loss = focal_binary_loss(logits[train_mask], yT[train_mask], swT[train_mask], gamma=FOCAL_GAMMA)
        else:
            loss = bce_weighted_loss(logits[train_mask], yT[train_mask], swT[train_mask], pos_w)
        opt.zero_grad(); loss.backward(); opt.step()

        # select by validation AUPR on full eval adjacency
        m.eval()
        with torch.no_grad():
            pv = torch.sigmoid(m(xT, A_hat)[val_mask]).cpu().numpy()
            yv = yT[val_mask].cpu().numpy()
            tau, f1v, aupr = pick_tau_on_val(yv, pv)

        if aupr > best_aupr:
            best_aupr = aupr
            best_state = {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
            best_info = dict(epoch=ep, tau=tau, f1=f1v, aupr=aupr)

    if best_state is not None:
        m.load_state_dict({k:v.to(DEVICE) for k,v in best_state.items()})
    return m, best_info

# test evaluation using PR-AUC 
def eval_test(m, A_hat_eval, tau):
    with torch.no_grad():
        p = torch.sigmoid(m(xT, A_hat_eval)).cpu().numpy()
    pr = (p >= tau).astype(int)
    y  = yT.cpu().numpy()
    te = np.where(test_mask.cpu().numpy())[0]
    acc = accuracy_score(y[te], pr[te])
    ap  = average_precision_score(y[te], p[te])    # PR-AUC (AP)
    f1p = f1_score(y[te], pr[te], pos_label=1)
    return acc, ap, f1p, pr, p

def fairness_DI_SPD(preds):
    rows = []
    test_idx = np.where(test_mask.cpu().numpy())[0]
    sw = pd.to_numeric(nodes.loc[test_idx, "instance_weight"], errors="coerce").fillna(0.0).to_numpy(float)
    for col, pretty in [("SEX_A","SEX_A"),("HISPALLP_BIN","HISPALLP_BIN")]:
        if col not in nodes.columns: continue
        g = pd.to_numeric(nodes.loc[test_idx, col], errors="coerce").fillna(0).astype(int).to_numpy()
        g0 = (g==0); g1 = (g==1)
        p0  = preds[test_idx][g0].mean() if g0.any() else np.nan
        p1  = preds[test_idx][g1].mean() if g1.any() else np.nan
        p0w = np.average(preds[test_idx][g0], weights=sw[g0]) if g0.any() and sw[g0].sum()>0 else np.nan
        p1w = np.average(preds[test_idx][g1], weights=sw[g1]) if g1.any() and sw[g1].sum()>0 else np.nan
        spd   = p0 - p1
        di    = (p0 / p1) if (p1 not in [0, np.nan]) else np.nan
        spd_w = p0w - p1w
        di_w  = (p0w / p1w) if (p1w not in [0, np.nan]) else np.nan
        rows.append({"attr":pretty, "SPD":spd, "DI":di, "SPD_w":spd_w, "DI_w":di_w})
    return pd.DataFrame(rows)

# compute train-weighted pos_weight once 
pi_w = float((yT[train_mask] * swT[train_mask]).sum().cpu().item() / max(swT[train_mask].sum().cpu().item(), 1e-12))
pos_w = (1.0 - pi_w) / max(pi_w, 1e-12)

# Baseline (PR-AUC) 
with torch.no_grad():
    pv = torch.sigmoid(model(xT, A_hat)[val_mask]).cpu().numpy()
    yv = yT[val_mask].cpu().numpy()
TAU_STAR, F1v, AUPRv = pick_tau_on_val(yv, pv)

acc0, ap0, f10, pr0, p0 = eval_test(model, A_hat, TAU_STAR)
base_fair = fairness_DI_SPD(pr0)
print("\nBaseline (no FairDrop): "
      f"Acc={acc0:.3f} PR-AUC={ap0:.3f} F1={f10:.3f} | "
      f"HISP DI_w={base_fair.loc[base_fair.attr=='HISPALLP_BIN','DI_w'].values[0]:.3f}, "
      f"SPD_w={base_fair.loc[base_fair.attr=='HISPALLP_BIN','SPD_w'].values[0]:+.3f}")

#  FairDrop sweep over p_same (HISP-only) 
rows = []
for p_same in [0.05, 0.10, 0.20]:
    # train on dropped adjacency (resampled per epoch inside)
    m, info = train_with_stochastic_fairdrop(p_same=p_same, epochs=EPOCHS)
    # evaluate on full adjacency
    acc, ap, f1p, pr, p = eval_test(m, A_hat, info['tau'])
    fair = fairness_DI_SPD(pr)
    rows.append({
        "method": f"FairDrop(HISP, p={p_same})",
        "val_AUPR": info['aupr'], "tau*": info['tau'],
        "Acc": acc, "PR-AUC": ap, "F1+": f1p,
        "DI_w(HISP)": float(fair.loc[fair.attr=="HISPALLP_BIN","DI_w"].values[0]),
        "SPD_w(HISP)": float(fair.loc[fair.attr=="HISPALLP_BIN","SPD_w"].values[0]),
    })

comp = pd.DataFrame(rows).round(3)
print("\n=== FairDrop(HISP-only) comparison (PR-AUC) ===")
print(comp.to_string(index=False))




Baseline (no FairDrop): Acc=0.467 PR-AUC=0.222 F1=0.329 | HISP DI_w=1.373, SPD_w=+0.200

=== FairDrop(HISP-only) comparison (PR-AUC) ===
                method  val_AUPR  tau*   Acc  PR-AUC   F1+  DI_w(HISP)  SPD_w(HISP)
FairDrop(HISP, p=0.05)     0.226 0.483 0.467   0.222 0.329       1.373          0.2
 FairDrop(HISP, p=0.1)     0.226 0.470 0.467   0.222 0.329       1.373          0.2
 FairDrop(HISP, p=0.2)     0.226 0.476 0.467   0.222 0.329       1.373          0.2


In [18]:
# Final Fairness Evaluation (SPD/DI + TPR/FPR parity) on TEST 
import numpy as np, pandas as pd, torch
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

#  Ensure probabilities / threshold exist (evaluate on the full A_hat)
if "probs_all" not in globals():
    model.eval()
    with torch.no_grad():
        logits = model(xT, A_hat)
    probs_all = torch.sigmoid(logits).cpu().numpy()

tau = globals().get("TAU_STAR", 0.5)
preds_all = (probs_all >= tau).astype(int)

#  Slice TEST
test_idx = np.where(test_mask.cpu().numpy())[0]
y_true = yT.cpu().numpy()[test_idx].astype(int)
y_pred = preds_all[test_idx].astype(int)
w_test = pd.to_numeric(nodes.loc[test_idx, "instance_weight"], errors="coerce").fillna(0.0).to_numpy(float)

#  Helpers
def aif_spd_di(attr_col, fav=1, unfav=0):
    """SPD/DI using AIF360. Mapping: 0=unprivileged, 1=privileged."""
    g = pd.to_numeric(nodes.loc[test_idx, attr_col], errors="coerce").fillna(0).astype(int).to_numpy()
    df_true = pd.DataFrame({attr_col: g, "label": y_true})
    df_pred = pd.DataFrame({attr_col: g, "label": y_pred})

    ds_true_u = BinaryLabelDataset(df=df_true, label_names=["label"],
                                   protected_attribute_names=[attr_col],
                                   favorable_label=fav, unfavorable_label=unfav)
    ds_pred_u = BinaryLabelDataset(df=df_pred, label_names=["label"],
                                   protected_attribute_names=[attr_col],
                                   favorable_label=fav, unfavorable_label=unfav)
    # weighted copies
    ds_true_w = ds_true_u.copy(); ds_true_w.instance_weights = w_test.copy()
    ds_pred_w = ds_pred_u.copy(); ds_pred_w.instance_weights = w_test.copy()

    cm_u = ClassificationMetric(ds_true_u, ds_pred_u,
                                unprivileged_groups=[{attr_col: 0}],
                                privileged_groups=[{attr_col: 1}])
    cm_w = ClassificationMetric(ds_true_w, ds_pred_w,
                                unprivileged_groups=[{attr_col: 0}],
                                privileged_groups=[{attr_col: 1}])

    return {
        "SPD_unw": cm_u.statistical_parity_difference(),     # unpriv - priv
        "DI_unw":  cm_u.disparate_impact(),                  # unpriv / priv
        "SPD_w":   cm_w.statistical_parity_difference(),
        "DI_w":    cm_w.disparate_impact(),
    }

def selection_rates(attr_col):
    """Selection rates P(ŷ=1|group) to interpret SPD/DI."""
    g = pd.to_numeric(nodes.loc[test_idx, attr_col], errors="coerce").fillna(0).astype(int).to_numpy()
    m0, m1 = (g==0), (g==1)
    p0  = y_pred[m0].mean() if m0.any() else np.nan
    p1  = y_pred[m1].mean() if m1.any() else np.nan
    p0w = np.average(y_pred[m0], weights=w_test[m0]) if m0.any() and w_test[m0].sum()>0 else np.nan
    p1w = np.average(y_pred[m1], weights=w_test[m1]) if m1.any() and w_test[m1].sum()>0 else np.nan
    return p0, p1, p0w, p1w

def error_rates(attr_col):
    """TPR/FPR (unweighted & weighted) per group."""
    g = pd.to_numeric(nodes.loc[test_idx, attr_col], errors="coerce").fillna(0).astype(int).to_numpy()
    out = {}
    for lab, mask in {"unpriv(0)": (g==0), "priv(1)": (g==1)}.items():
        pos = mask & (y_true==1)
        neg = mask & (y_true==0)
        # unweighted
        tpr  = y_pred[pos].mean() if pos.any() else np.nan
        fpr  = y_pred[neg].mean() if neg.any() else np.nan
        # weighted
        tpr_w = (np.average(y_pred[pos], weights=w_test[pos]) if pos.any() and w_test[pos].sum()>0 else np.nan)
        fpr_w = (np.average(y_pred[neg], weights=w_test[neg]) if neg.any() and w_test[neg].sum()>0 else np.nan)
        out[lab] = {"TPR": tpr, "FPR": fpr, "TPR_w": tpr_w, "FPR_w": fpr_w}
    return pd.DataFrame(out).round(3)

#  Run for both protected attributes (defined 0=unpriv, 1=priv; favorable=1 by design)
rows = []
for col, label in [
    ("SEX_A",        "SEX_A (0=Male unpriv, 1=Female priv)"),
    ("HISPALLP_BIN", "HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)")
]:
    if col not in nodes.columns: continue
    spddi = aif_spd_di(col, fav=1, unfav=0)
    p0, p1, p0w, p1w = selection_rates(col)
    rows.append({
        "attr": label,
        "P1_unpriv": round(p0, 3), "P1_priv": round(p1, 3),
        "P1_unpriv_w": round(p0w, 3), "P1_priv_w": round(p1w, 3),
        "SPD_unw": round(spddi["SPD_unw"], 3),
        "DI_unw":  round(spddi["DI_unw"], 3) if np.isfinite(spddi["DI_unw"]) else np.nan,
        "SPD_w":   round(spddi["SPD_w"], 3),
        "DI_w":    round(spddi["DI_w"], 3) if np.isfinite(spddi["DI_w"]) else np.nan,
        "4/5ths_ok_unw": (0.8 <= spddi["DI_unw"] <= 1.25) if np.isfinite(spddi["DI_unw"]) else np.nan,
        "4/5ths_ok_w":   (0.8 <= spddi["DI_w"]   <= 1.25) if np.isfinite(spddi["DI_w"])   else np.nan,
    })

fair_tbl = pd.DataFrame(rows)
print("=== AIF360 SPD / DI (Test) with your privilege mapping (fav=1) ===")
print(fair_tbl.to_string(index=False))

print("\n=== Error-rate parity (TPR/FPR) on TEST ===")
print("SEX_A:\n", error_rates("SEX_A").to_string())
print("\nHISPALLP_BIN:\n", error_rates("HISPALLP_BIN").to_string())


=== AIF360 SPD / DI (Test) with your privilege mapping (fav=1) ===
                                                attr  P1_unpriv  P1_priv  P1_unpriv_w  P1_priv_w  SPD_unw  DI_unw  SPD_w  DI_w  4/5ths_ok_unw  4/5ths_ok_w
                SEX_A (0=Male unpriv, 1=Female priv)      0.624    0.598        0.625      0.586    0.026   1.043  0.039 1.066           True         True
HISPALLP_BIN (0=Hisp+Others unpriv, 1=NH White priv)      0.734    0.545        0.735      0.535    0.188   1.345  0.200 1.373          False        False

=== Error-rate parity (TPR/FPR) on TEST ===
SEX_A:
        unpriv(0)  priv(1)
TPR        0.685    0.720
FPR        0.615    0.561
TPR_w      0.701    0.728
FPR_w      0.610    0.555

HISPALLP_BIN:
        unpriv(0)  priv(1)
TPR        0.789    0.682
FPR        0.725    0.509
TPR_w      0.783    0.679
FPR_w      0.725    0.507


In [19]:
# homophily on FairDrop-perturbed training adj (HISP-only), averaged over K samples
import numpy as np, pandas as pd, torch
from scipy.sparse import coo_matrix

DEVICE = next(model.parameters()).device
K = 10          # number of resamples to average
P_SAME = 0.20   # same-group drop prob 

# helper to build one sampled training adjacency (same as FairDrop sampler)
def sample_train_adj_hisp(p_same):
    g = torch.from_numpy(pd.to_numeric(nodes["HISPALLP_BIN"], errors="coerce").fillna(0).astype(int).to_numpy()).to(DEVICE)
    src, dst = eiT[0], eiT[1]     # edge index from GCN prep
    same = (g[src] == g[dst])
    keep_prob = torch.where(same, torch.tensor(1.0 - p_same, device=DEVICE), torch.tensor(1.0, device=DEVICE))
    keep = torch.rand_like(ewT) < keep_prob
    ei = eiT[:, keep].cpu().numpy(); ew = ewT[keep].cpu().numpy()
    n = A_s.shape[0]
    return coo_matrix((ew, (ei[0], ei[1])), shape=(n, n)).tocsr()

def homophily_obs_exp(A_csr, group01):
    from scipy.sparse import triu
    U = triu(A_csr, k=1).tocoo()
    if U.nnz == 0: return np.nan, np.nan
    same = (group01[U.row] == group01[U.col])
    obs = float(U.data[same].sum() / (U.data.sum() + 1e-12))
    endp = np.concatenate([group01[U.row], group01[U.col]])
    p = np.bincount(endp, minlength=2).astype(float); p /= p.sum()
    exp = float((p**2).sum())
    return obs, exp

g_hisp = nodes["HISPALLP_BIN"].to_numpy(int)
obs_list, exp_list = [], []
for _ in range(K):
    A_fd = sample_train_adj_hisp(P_SAME)
    A_fd = ((A_fd + A_fd.T) * 0.5).tocsr()
    o,e = homophily_obs_exp(A_fd, g_hisp)
    obs_list.append(o); exp_list.append(e)

print(f"HISP homophily on base graph:    obs/exp = {homophily_from_A(A_s, g_hisp)[0]:.3f}/{homophily_from_A(A_s, g_hisp)[1]:.3f}")
print(f"HISP homophily with FairDrop p={P_SAME:.2f}: mean obs/exp over {K} samples = {np.nanmean(obs_list):.3f}/{np.nanmean(exp_list):.3f}")


HISP homophily on base graph:    obs/exp = 0.567/0.550
HISP homophily with FairDrop p=0.20: mean obs/exp over 10 samples = 0.512/0.549


In [1]:
#https://github.com/parisots/population-gcn
#https://medium.com/@kaoningyu/introduction-of-graph-convolutional-network-gcn-quick-implementation-5dd75e75b261
#https://www.datacamp.com/tutorial/comprehensive-introduction-graph-neural-networks-gnns-tutorial
#https://www.geeksforgeeks.org/deep-learning/graph-convolutional-networks-gcns-architectural-insights-and-applications/
#https://youtu.be/G6c6zk0RhRM?si=zhYWFjP8_Ev3Brwv - Graph Convolutional Networks (GCNs) in PyTorch
#https://youtu.be/PQT2QblNegY?si=eohnLdzibHvpEdYR - PyTorch code for GCN and SGC
#https://youtu.be/8qTnNXdkF1Q?si=6DvTrpHklrvCNq_A  - Graph Convolutional Networks using only NumPy
#https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.assortativity.attribute_assortativity_coefficient.html
#https://graph-tool.skewed.de/static/docs/stable/autosummary/graph_tool.correlations.assortativity.html
#https://pytorch-geometric.readthedocs.io/en/2.1.0/_modules/torch_geometric/utils/homophily.html